In [1]:
# 1. 기본 설정
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
# 2. train_v2 + members_v3 로드 (01_eda.ipynb와 동일한 방식)
train = pd.read_csv("../data/train_v2.csv")

members = pd.read_csv("../data/members_v3.csv")
target_users = set(train["msno"])
members = members[members["msno"].isin(target_users)].copy()
members["bd_clean"] = members["bd"].where(members["bd"].between(10, 80))

print(train.shape, members.shape)

(970960, 2) (860967, 7)


In [3]:
# 3. [핵심] 관측 시점 기준으로 안전한 거래 피처 생성
# -> 01_eda.ipynb에서 발견한 누수: "마지막 거래"는 이탈/갱신을 결정짓는 사건 그 자체라 그대로 쓰면 정답을 미리 보는 셈
# -> 해결: 사용자별로 시간순 정렬 후 "마지막 거래 1건을 제외"하고, 그 이전 이력만으로 피처를 만든다
#    (마지막 거래 자체가 아니라 "그 직전까지의 행동 패턴"으로 이탈을 예측하는 것이 올바른 설계)
transactions = pd.read_parquet("../data/transactions_filtered.parquet")
transactions = transactions.sort_values("transaction_date")
transactions["rank_desc"] = transactions.groupby("msno").cumcount(ascending=False)

history = transactions[transactions["rank_desc"] > 0]  # 마지막 거래 제외한 사전 이력
print("전체 거래:", len(transactions), "/ 사전이력만:", len(history))

전체 거래: 16255622 / 사전이력만: 15284662


In [4]:
# 4. 사전 이력을 고객별 요약 피처로 집계
# -> tx_count: 사전 결제 횟수 / total_paid: 사전 총 결제금액 / cancel_count: 사전 해지신청 횟수
# -> prior_auto_renew, prior_plan_days: 마지막 거래 "직전" 거래 기준 자동갱신 여부/요금제 기간
agg = (
    history.groupby("msno")
    .agg(
        tx_count=("transaction_date", "count"),
        total_paid=("actual_amount_paid", "sum"),
        avg_plan_days=("payment_plan_days", "mean"),
        cancel_count=("is_cancel", "sum"),
        prior_auto_renew=("is_auto_renew", "last"),
        prior_plan_days=("payment_plan_days", "last"),
    )
    .reset_index()
)
print(agg.shape)
agg.describe()

(963293, 7)


,tx_count,total_paid,avg_plan_days,cancel_count,prior_auto_renew,prior_plan_days
count,963293.000000,963293.000000,963293.000000,963293.000000,963293.000000,963293.000000
mean,15.867095,2166.262791,33.242179,0.260280,0.888780,34.082403
std,8.657558,1288.463264,29.718169,0.569354,0.314404,33.488786
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,8.000000,1044.000000,28.888889,0.000000,1.000000,30.000000
50%,16.000000,1881.000000,30.000000,0.000000,1.000000,30.000000
75%,23.000000,3278.000000,30.000000,0.000000,1.000000,30.000000
max,243.000000,16590.000000,450.000000,21.000000,1.000000,450.000000


In [5]:
# 5. [핵심] 관측 시점 기준으로 안전한 청취 활동 피처 생성 (user_logs.csv + user_logs_v2.csv)
# -> user_logs는 30GB가 넘어 한 번에 메모리에 올릴 수 없으므로, 청크 단위로 읽으면서
#    (1) 대상 고객만 필터링 (2) 각 고객의 "마지막 거래일 이전" 로그만 사용 (3) 부분 집계를 바로 계산하고,
#    마지막에 부분 집계들만 합산하는 스트리밍 방식으로 메모리 문제 없이 전체를 처리한다 (최초 1회 15~20분 소요)
# -> total_secs 컬럼에는 원본 데이터 자체의 오류로 음수/수천조 단위의 손상된 값이 일부(약 0.05%) 섞여 있어,
#    하루 최대치(86400초)를 벗어나는 값은 집계에서 제외한다
import os

LOG_FEATURES_CACHE = "../data/user_logs_features.parquet"
last_tx_date = transactions.groupby("msno")["transaction_date"].max().rename("last_transaction_date")
SONG_COLS = ["num_25", "num_50", "num_75", "num_985", "num_100"]

if os.path.exists(LOG_FEATURES_CACHE):
    log_features = pd.read_parquet(LOG_FEATURES_CACHE)
else:
    log_partials = []
    for fname in ["../data/user_logs.csv", "../data/user_logs_v2.csv"]:
        for chunk in pd.read_csv(fname, chunksize=5_000_000):
            chunk = chunk[chunk["msno"].isin(target_users)]
            if len(chunk) == 0:
                continue
            chunk = chunk.join(last_tx_date, on="msno")
            chunk = chunk[chunk["date"] < chunk["last_transaction_date"]]
            if len(chunk) == 0:
                continue
            chunk = chunk.copy()
            chunk["total_songs"] = chunk[SONG_COLS].sum(axis=1)
            chunk["total_secs_clean"] = chunk["total_secs"].where(chunk["total_secs"].between(0, 86400))
            partial = chunk.groupby("msno").agg(
                log_days=("date", "count"),
                total_secs_sum=("total_secs_clean", "sum"),
                num_unq_sum=("num_unq", "sum"),
                num_100_sum=("num_100", "sum"),
                total_songs_sum=("total_songs", "sum"),
            )
            log_partials.append(partial)

    log_agg = pd.concat(log_partials).groupby("msno").sum().reset_index()
    log_agg["avg_secs_per_day"] = log_agg["total_secs_sum"] / log_agg["log_days"]
    log_agg["avg_unq_per_day"] = log_agg["num_unq_sum"] / log_agg["log_days"]
    log_agg["completion_ratio"] = log_agg["num_100_sum"] / log_agg["total_songs_sum"].replace(0, np.nan)
    log_features = log_agg[["msno", "log_days", "avg_secs_per_day", "avg_unq_per_day", "completion_ratio"]]
    log_features.to_parquet(LOG_FEATURES_CACHE)

print(log_features.shape)
log_features.describe()

(850465, 5)


,log_days,avg_secs_per_day,avg_unq_per_day,completion_ratio
count,850465.000000,850465.000000,850465.000000,850465.000000
mean,285.282946,6558.719589,25.202161,0.679121
std,229.938736,5301.495065,17.294358,0.197319
min,1.000000,0.000000,1.000000,0.000000
25%,80.000000,3422.112609,14.394915,0.578801
50%,235.000000,5248.594729,21.391061,0.715915
75%,460.000000,7995.568955,31.212987,0.822623
max,820.000000,82132.440837,1308.240000,1.000000


In [6]:
# 6. train + members + 누수 없는 거래 피처 + 누수 없는 청취 활동 피처 병합, 신규 고객 플래그 추가
# -> 사전 이력이 0건인 고객(7,667명, 0.8%)은 총 거래가 1건뿐이라는 뜻 -> 별도로 "신규 고객" 플래그 부여
df = train.merge(members, on="msno", how="left").merge(agg, on="msno", how="left").merge(log_features, on="msno", how="left")
df["is_new_customer"] = df["tx_count"].isnull().astype(int)
print(df.shape)
print("신규 고객 수:", df["is_new_customer"].sum())
print()
print("[검증] prior_auto_renew별 이탈률 (누수 제거 후에도 신호가 유효한지 재확인):")
print(df.groupby("prior_auto_renew")["is_churn"].agg(["mean", "count"]))
print()
print("[검증] 신규 고객 여부별 이탈률:")
print(df.groupby("is_new_customer")["is_churn"].agg(["mean", "count"]))
print()
print("[검증] 관측 시점 이전 청취 기록 유무별 이탈률 (청취 활동 피처가 유효한 신호인지 확인):")
has_logs = df["log_days"].notnull()
print(df.groupby(has_logs)["is_churn"].agg(["mean", "count"]))
print()
print("[검증] 완청 비율(completion_ratio) 구간별 이탈률 (청취 기록이 있는 고객만):")
has_ratio = df["completion_ratio"].notnull()
ratio_bin = pd.qcut(df.loc[has_ratio, "completion_ratio"], 4, duplicates="drop")
print(df.loc[has_ratio].groupby(ratio_bin, observed=True)["is_churn"].agg(["mean", "count"]))

(970960, 19)
신규 고객 수: 7667

[검증] prior_auto_renew별 이탈률 (누수 제거 후에도 신호가 유효한지 재확인):
                      mean   count
prior_auto_renew                  
0.0               0.381885  107137
1.0               0.046706  856156

[검증] 신규 고객 여부별 이탈률:
                     mean   count
is_new_customer                  
0                0.083985  963293
1                0.838398    7667

[검증] 관측 시점 이전 청취 기록 유무별 이탈률 (청취 활동 피처가 유효한 신호인지 확인):
              mean   count
log_days                  
False     0.077804  120495
True      0.091662  850465

[검증] 완청 비율(completion_ratio) 구간별 이탈률 (청취 기록이 있는 고객만):


                      mean   count
completion_ratio                  
(-0.001, 0.579]   0.092584  212617
(0.579, 0.716]    0.093581  212617
(0.716, 0.823]    0.091790  212615
(0.823, 1.0]      0.088690  212616


In [7]:
# 7. 결측치 처리
# -> 거래 이력 피처(tx_count 등): 신규 고객은 사전 이력이 없어 결측 -> 0으로 채움 (is_new_customer로 구분은 유지됨)
# -> log_days: 관측 시점 이전 청취 기록이 없는 고객 -> 0으로 채움 (활동이 없었다는 것 자체가 의미 있는 값)
# -> avg_secs_per_day, avg_unq_per_day, completion_ratio: log_days=0이면 정의 자체가 불가능한 비율이므로
#    0으로 채우지 않고 결측으로 남겨 모델링 파이프라인의 중앙값 대체(SimpleImputer)에 맡긴다 (bd_clean과 동일한 방식)
# -> city, registered_via: members 조인 실패 고객(11만 명) -> 문자열 "-1"이라는 별도 범주로 처리
# -> gender: 결측 54.8% -> "unknown"이라는 별도 범주로 처리 (더미변수라 결측 자체가 정보가 될 수 있음)
for col in ["tx_count", "total_paid", "avg_plan_days", "cancel_count", "prior_auto_renew", "prior_plan_days", "log_days"]:
    df[col] = df[col].fillna(0)

df["city"] = df["city"].fillna(-1).astype(str)
df["registered_via"] = df["registered_via"].fillna(-1).astype(str)
df["gender"] = df["gender"].fillna("unknown").astype(str)

print(df.isnull().sum())

msno                           0
is_churn                       0
city                           0
bd                        109993
gender                         0
registered_via                 0
registration_init_time    109993
bd_clean                  584567
tx_count                       0
total_paid                     0
avg_plan_days                  0
cancel_count                   0
prior_auto_renew               0
prior_plan_days                0
log_days                       0
avg_secs_per_day          120495
avg_unq_per_day           120495
completion_ratio          120495
is_new_customer                0
dtype: int64


In [8]:
# 8. 피처/타겟 정의 및 train/test 분할
# -> bd_clean, avg_secs_per_day, avg_unq_per_day, completion_ratio는 결측이 남아있는 상태로 두고,
#    03_modeling.ipynb의 전처리 파이프라인(SimpleImputer)에서 처리
from sklearn.model_selection import train_test_split

NUMERIC = [
    "bd_clean", "tx_count", "total_paid", "avg_plan_days", "cancel_count", "prior_plan_days",
    "log_days", "avg_secs_per_day", "avg_unq_per_day", "completion_ratio",
]
CATEGORICAL = ["city", "gender", "registered_via"]
BINARY = ["prior_auto_renew", "is_new_customer"]

X = df[NUMERIC + CATEGORICAL + BINARY]
y = df["is_churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)

(776768, 15) (194192, 15)


In [9]:
# 9. 전처리 결과 저장 (03_modeling.ipynb에서 이어서 사용)
train_processed = X_train.copy()
train_processed["is_churn"] = y_train
test_processed = X_test.copy()
test_processed["is_churn"] = y_test

train_processed.to_parquet("../data/train_processed.parquet")
test_processed.to_parquet("../data/test_processed.parquet")
print("저장 완료: data/train_processed.parquet, data/test_processed.parquet")

저장 완료: data/train_processed.parquet, data/test_processed.parquet
